In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = '/content/drive/My Drive/tire_extraction/yolo_experiment1'
os.listdir(project_path)

['resnet50-oclip.pth',
 'yolo11n.yml',
 'coco.yaml',
 'yolo11n-seg.pt',
 'ultralytics_src.zip',
 'outputs',
 'yolov11n-seg_new.pt',
 'y5s_c3f8.zip',
 'yolov5.zip',
 'yolov11n_seg.yaml',
 'yolov11n_oclip_transfer.pt',
 'backbone_transfer.py',
 'ultralytics.zip']

In [ ]:
destination_dir = '/content/'

In [ ]:
!cp -r "$project_path"/* "$destination_dir" &>/dev/null

print(f"Files from '{project_path}' copied to '{destination_dir}'")

Files from '/content/drive/My Drive/tire_extraction/yolo_experiment1' copied to '/content/'


In [ ]:
zip_file_path = '/content/drive/My Drive/tire_extraction/data/yolo_tire.zip'
extract_dir = '/content/dataset/'

if not os.path.exists(extract_dir):
    os.makedirs(extract_dir)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Unzipped {zip_file_path} to {extract_dir}")
!ls {extract_dir}

Unzipped /content/drive/My Drive/tire_extraction/data/yolo_tire.zip to /content/dataset/
images	labels


In [ ]:
!python -m pip install /content/ultralytics.zip

Processing ./ultralytics.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.179-py3-none-any.whl size=1044910 sha256=b029e6855c0aff58755577028ca528dd5714516a01b9d654374d36d63f8c86de
  Stored in directory: /tmp/pip-ephem-wheel-cache-w8u4ghss/wheels/3a/2d/70/3511dfa0be67c4ecac1d6860b781bf0fdf5623d4f426bd2f64
Successfully built ultralytics


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!python3 /content/backbone_transfer.py

OClip (ResNet-vd) -> YOLO11 Optimized Backbone Transfer
Loading checkpoint from /content/resnet50-oclip.pth...
✓ Loaded 330 keys after prefix stripping.
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Processing Stem (OClipStem)...
  -> Stem conv1: stem.0.weight -> Conv (3, 3)
  -> Stem conv2: stem.3.weight -> Conv (3, 3)
  -> Stem conv3: stem.6.weight -> Conv (3, 3)
Detected ResNet stages at backbone indices: [1, 2, 3, 4]
Processing Stage idx=1 -> CLIP layer1 (3 blocks)
Processing Stage idx=2 -> CLIP layer2 (4 blocks)
Processing Stage idx=3 -> CLIP layer3 (6 blocks)
Processing Stage idx=4 -> CLIP layer4 (3 blocks)

✓ Saved transferred model to: /content/yolov11n_transform.pt
Note: Asp4Layers and any extra blocks without CLIP counterparts stay randomly initialized.


In [ ]:
model = YOLO("/content/yolov11n_transform.pt")

In [ ]:
for i, m in enumerate(model.model.model):
    class_name = m.__class__.__name__

    if class_name in ['OClipStem', 'OClipResLayer']:
        print(f"Freezing {class_name} at index {i}")
        for p in m.parameters():
            p.requires_grad = False

    elif class_name == 'Asp4Layers':
        print(f"Keeping {class_name} trainable at index {i}")

model.train(
    task="segment",
    data="coco.yaml",
    epochs=100,
    batch=16,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    amp=True,
    project='/content/drive/My Drive/tire_extraction/yolo_experiment1/outputs',
    name="yolov11n_oclip_asp_p3"
)


Freezing OClipStem at index 0
Freezing OClipResLayer at index 1
Freezing OClipResLayer at index 2
Freezing OClipResLayer at index 3
Freezing OClipResLayer at index 4
Keeping Asp4Layers trainable at index 11
New https://pypi.org/project/ultralytics/8.3.234 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.179 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=coco.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0

WARNING ⚠️ no model scale passed. Assuming scale='n'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.custom.OClipStem      [3, 16]                       
  1                  -1  1     14016  ultralytics.nn.modules.custom.OClipResLayer  [16, 16, 3, 1]                
  2                  -1  1     77568  ultralytics.nn.modules.custom.OClipResLayer  [64, 32, 4, 2]                
  3                  -1  1    447488  ultralytics.nn.modules.custom.OClipResLayer  [128, 64, 6, 2]               
  4                  -1  1    939520  ultralytics.nn.modules.custom.OClipResLayer  [256, 128, 3, 2]              
  5                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
  6             [-1, 3]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
  7                  -1  1    160

YOLOv11n_seg summary: 286 layers, 3,474,403 parameters, 3,474,387 gradients, 17.4 GFLOPs

Transferred 711/711 items from pretrained weights
Freezing layer 'model.18.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2779.0±963.1 MB/s, size: 144.2 KB)


train: Scanning /content/dataset/labels/train... 4390 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4390/4390 [00:03<00:00, 1354.99it/s]


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   3%|▎         | 1/35 [00:00<00:08,  4.23it/s]

WARNING ⚠️ Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 35/35 [00:09<00:00,  3.62it/s]


                   all       1098       2140      0.928      0.902      0.958      0.723       0.93      0.903       0.96      0.703
Speed: 0.2ms preprocess, 2.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/drive/My Drive/tire_extraction/yolo_experiment1/outputs/yolov11n_oclip_asp_p3


ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bc195202720>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [ ]:
ffffd;fdfdfdxcdfdfddsdfdfdvvfdgfgffdfdfjjjdffdffdgfgffdfdfdfd

In [ ]:
lklksd